# Learning regulatory DNA motifs with a 1D CNN

This notebook is the portfolio entry point. It reproduces the core workflow from the repository: generate a controlled DNA benchmark, one-hot encode sequences, train a CNN, evaluate it, and inspect one sequence with in-silico mutagenesis.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve() / 'src'))

import torch
from dataset import make_dataset, one_hot
from model import DNAConvNet


## 1. Generate the benchmark

Positive examples contain a REST-like motif sampled from the JASPAR MA0138.2 profile. Negative examples are random DNA. This is a controlled benchmark, not a biological prediction dataset.

In [ ]:
seqs, y = make_dataset(n=4000, seq_len=200, seed=42)
X = torch.from_numpy(one_hot(seqs))
Y = torch.from_numpy(y)
X.shape, Y.shape


The expected input shape is `[batch, channels, length] = [B, 4, 200]`. The four channels correspond to A/C/G/T.

In [ ]:
model = DNAConvNet()
print(model)
with torch.no_grad():
    logits = model(X[:8])
logits.shape


## 2. Training concept

The repository's training script uses `BCEWithLogitsLoss` and Adam. The essential loop is:

1. forward pass
2. compute loss
3. `loss.backward()`
4. `optimizer.step()`

This is the core mechanism behind supervised deep learning.

In [ ]:
# Reproduce the full experiment from the terminal with:
# python src/train.py --n-samples 4000 --epochs 5 --batch-size 128 --cpu --out-dir results


## 3. Interpretation

After training, `src/interpret.py` changes one base at a time and measures the change in the prediction. Large absolute changes identify positions to inspect further. This is model sensitivity, not proof of biological causality.